# House Price Prediction Project

This notebook trains two models:

- **Current Price Prediction** using regression
- **Good Investment Prediction** using classification

It also prepares the data that is used in the Streamlit app.


In [ ]:
import json
import os
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder

In [ ]:
# Paths
DATA_PATH = "india_housing_prices(3).csv"
MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)

SAMPLE_SIZE = 100000  # set to None to use the full dataset

df = pd.read_csv(DATA_PATH)

if SAMPLE_SIZE is not None and SAMPLE_SIZE < len(df):
    df = df.sample(SAMPLE_SIZE, random_state=42).reset_index(drop=True)

print("Rows:", len(df))
print("Columns:", df.shape[1])
df.head()

## Feature Engineering

We use the useful input columns from the dataset and create a few extra helper features like property age, size per BHK, floor ratio, amenity count, and amenity flags.


In [ ]:
TOP_AMENITIES = ["Garden", "Pool", "Clubhouse", "Gym", "Playground"]

def add_features(data: pd.DataFrame) -> pd.DataFrame:
    df = data.copy()

    # Basic engineered features
    df["Age_of_Property_Calc"] = (2026 - df["Year_Built"]).clip(lower=0)
    df["Size_per_BHK"] = df["Size_in_SqFt"] / df["BHK"].clip(lower=1)
    df["Floor_Ratio"] = df["Floor_No"] / df["Total_Floors"].clip(lower=1)
    df["Is_New"] = (df["Age_of_Property_Calc"] <= 5).astype(int)
    df["Is_Large"] = (df["Size_in_SqFt"] >= 1200).astype(int)

    # Amenities features
    amenity_text = df["Amenities"].fillna("").astype(str)

    df["Amenity_Count"] = amenity_text.apply(
        lambda x: 0 if not x.strip() else len([a.strip() for a in x.split(",") if a.strip()])
    )

    for amenity in TOP_AMENITIES:
        df[f"Amenity_{amenity}"] = amenity_text.str.contains(amenity, regex=False).astype(int)

    # Good investment target for classification
    df["ROI"] = df["Price_in_Lakhs"] / df["Size_in_SqFt"].clip(lower=1)
    df["Good_Investment"] = (df["ROI"] <= df["ROI"].median()).astype(int)

    # 5-year future price target (derived from projected annual growth)
    projected_growth = np.full(len(df), 0.045, dtype=float)

    projected_growth += np.where(df["Amenity_Count"] >= 4, 0.02, 0.0)
    projected_growth += np.where((df["Amenity_Count"] >= 2) & (df["Amenity_Count"] < 4), 0.01, 0.0)
    projected_growth += np.where(df["Nearby_Schools"] >= 5, 0.01, 0.0)
    projected_growth += np.where(df["Nearby_Hospitals"] >= 3, 0.01, 0.0)
    projected_growth += np.where(df["Is_New"] == 1, 0.015, 0.0)
    projected_growth += np.where(df["Age_of_Property_Calc"] >= 20, -0.01, 0.0)

    df["Projected_Growth_Rate"] = np.clip(projected_growth, 0.03, 0.15)
    df["Future_Price_5Y"] = df["Price_in_Lakhs"] * ((1 + df["Projected_Growth_Rate"]) ** 5)

    return df

df = add_features(df)
df.head()


## Targets

- `Price_in_Lakhs` is the **current price target**.
- `Good_Investment` is a **classification target** created from ROI.
- `Future_Price_5Y` is a **derived regression target** for the estimated price after 5 years.

We do **not** use `Price_per_SqFt` as a feature because it is derived from the target and can leak information.


In [ ]:
numeric_features = [
    "BHK",
    "Size_in_SqFt",
    "Year_Built",
    "Floor_No",
    "Total_Floors",
    "Nearby_Schools",
    "Nearby_Hospitals",
    "Age_of_Property_Calc",
    "Size_per_BHK",
    "Floor_Ratio",
    "Amenity_Count",
    "Amenity_Garden",
    "Amenity_Pool",
    "Amenity_Clubhouse",
    "Amenity_Gym",
    "Amenity_Playground",
    "Is_New",
    "Is_Large",
]

categorical_features = [
    "State",
    "City",
    "Locality",
    "Property_Type",
    "Furnished_Status",
    "Public_Transport_Accessibility",
    "Parking_Space",
    "Security",
    "Facing",
    "Owner_Type",
    "Availability_Status",
]

feature_columns = numeric_features + categorical_features

X = df[feature_columns].copy()
y_reg = df["Price_in_Lakhs"].copy()
y_cls = df["Good_Investment"].copy()
y_future = df["Future_Price_5Y"].copy()

print("Feature count:", len(feature_columns))
print("X shape:", X.shape)
print("Regression target shape:", y_reg.shape)
print("5-year future price target shape:", y_future.shape)
print("Classification target distribution:")
print(y_cls.value_counts(normalize=True).round(3))


## Train-Test Split

We use the same features for both tasks.


In [ ]:
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X, y_reg, test_size=0.2, random_state=42
)

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X, y_cls, test_size=0.2, random_state=42, stratify=y_cls
)

print("Regression train:", X_train_r.shape, "test:", X_test_r.shape)
print("Classification train:", X_train_c.shape, "test:", X_test_c.shape)

## Preprocessing + Models

We use ordinal encoding for categorical features and Random Forest models for both outputs.


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median"))
            ]),
            numeric_features
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
            ]),
            categorical_features
        ),
    ],
    remainder="drop"
)

price_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestRegressor(
            n_estimators=120,
            max_depth=18,
            random_state=42,
            n_jobs=-1
        ))
    ]
)

investment_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=120,
            max_depth=18,
            random_state=42,
            n_jobs=-1
        ))
    ]
)

## Training


In [ ]:
price_model.fit(X_train_r, y_train_r)
investment_model.fit(X_train_c, y_train_c)

print("Training completed.")

## Evaluation


In [ ]:
price_pred = price_model.predict(X_test_r)
invest_pred = investment_model.predict(X_test_c)

print("=== Current Price Model ===")
print("MAE:", round(mean_absolute_error(y_test_r, price_pred), 3))
print("R2 :", round(r2_score(y_test_r, price_pred), 3))

print("
=== Investment Model ===")
print("Accuracy:", round(accuracy_score(y_test_c, invest_pred), 3))
print(classification_report(y_test_c, invest_pred))

## Save Models + Metadata


In [ ]:
metadata = {
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "top_amenities": TOP_AMENITIES,
    "feature_columns": feature_columns,
}

joblib.dump(price_model, MODEL_DIR / "price_model.pkl", compress=3)
joblib.dump(investment_model, MODEL_DIR / "investment_model.pkl", compress=3)

with open(MODEL_DIR / "house_price_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print("Saved:")
print(" - models/price_model.pkl")
print(" - models/investment_model.pkl")
print(" - models/house_price_metadata.json")

## Example: 5-Year Future Price and Investment Idea

The future price is not a dataset target, so we estimate it from the predicted current price using a feature-based growth rate.


In [ ]:
def estimate_growth_rate(row: pd.Series) -> float:
    age = int(row["Age_of_Property_Calc"])
    amenity_count = int(row["Amenity_Count"])

    transport_map = {"Low": 0.00, "Medium": 0.01, "High": 0.02}
    yesno_map = {"No": 0.0, "Yes": 0.01}

    rate = 0.045
    rate += transport_map.get(row["Public_Transport_Accessibility"], 0.0)
    rate += yesno_map.get(row["Security"], 0.0)
    rate += yesno_map.get(row["Parking_Space"], 0.0)

    if amenity_count >= 4:
        rate += 0.02
    elif amenity_count >= 2:
        rate += 0.01

    if age <= 5:
        rate += 0.015
    elif age >= 20:
        rate -= 0.01

    return float(np.clip(rate, 0.03, 0.15))

sample_row = X_test_r.iloc[0]
current_price = float(price_model.predict(pd.DataFrame([sample_row]))[0])
growth_rate = estimate_growth_rate(df.loc[sample_row.name])
future_price = current_price * ((1 + growth_rate) ** 5)
investment_prob = float(investment_model.predict_proba(pd.DataFrame([sample_row]))[0, 1])

print("Current Price (Lakhs):", round(current_price, 2))
print("Estimated 5-Year Price (Lakhs):", round(future_price, 2))
print("Expected Growth %:", round((future_price / current_price - 1) * 100, 2))
print("Investment Probability:", round(investment_prob * 100, 2))